# ICOS atmospheric NRT — two species at once

**Experimental variant.** The standard NRT notebook works on one species at a time. This one loads
**two** — CO against CO₂, for instance — for the same station, and merges them on the time column so
that they can be compared directly.

Configuration, dependencies and station choice are the same cells as
`ICOS_ex1a_atmo_data_NRT.ipynb`.

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* before running the notebook
> again and before saving it: widget state belongs to the kernel that created it, and leftovers
> make the controls stop responding.

## 0. Configuration

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook
NOTEBOOK_NAME = "ICOS atmospheric NRT - two species"
NOTEBOOK_VERSION = "0.1"
NOTEBOOK_DATE = "2026-09-15"

DEFAULT_COUNTRY = "IT"
DEFAULT_STATION_CODE = "CMN"
# The two NRT products to compare. Written as the species codes appear in the data-type URIs.
DEFAULT_GASES = ["CO", "CO2"]

DEFAULT_QUALITY_FLAG = "Flag"
# NRT data carry "U" (unchecked): the quality control has not run yet, which is the point of a
# near real time product. "O" (ok) is what the Level 2 products use, and filtering NRT on it
# discards every row. Both are accepted here; set ACCEPTED_FLAGS = ("O",) to keep only data
# that has been through quality control, which for NRT usually means nothing at all.
ACCEPTED_FLAGS = ("O", "U")
MAP_BACKEND = "plotly"
MAX_FILES_PER_GAS = 30
# When the two products do not share exact timestamps, each point is matched with the nearest
# one within this window instead of being discarded.
ALIGN_TOLERANCE = "30min"
TIME_COLUMN_CANDIDATES = ("TIMESTAMP", "SamplingStart", "TimeStamp", "time", "Date")

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

## 1. Dependencies and imports

In [ ]:
import os
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    "icoscp_core": "icoscp",
    # Hidden dependency: plotly needs nbformat to render figures, but does not declare it.
    "nbformat": "nbformat",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                   check=False, cwd=os.path.expanduser("~"))
else:
    print("All required packages are already installed.")

import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode

# Loads the Plotly JavaScript into the page. Without it, Plotly figures shown inside an
# ipywidgets Output (the station map in section 2) render as an empty area. connected=True
# pulls the library from the CDN instead of embedding ~5 MB into the saved notebook.
if MAP_BACKEND == "plotly":
    init_notebook_mode(connected=True)
import ipywidgets as widgets
from IPython.display import display

from icoscp_core.icos import data, meta, ATMO_STATION


# --- Outputs of this run -------------------------------------------------------------
# Figures are shown inline and kept in memory; nothing is written to disk while you work.
# The export cell at the end of the notebook writes out what you decide to keep, into the
# folder below, and packs it into a single zip you can download.
#
# The folder is emptied at the start of each run, so that repeated runs do not accumulate
# files. Set CLEAN_OUTPUTS to False to keep what earlier runs produced.
import shutil
from pathlib import Path

OUTPUT_DIR = Path("outputs")
CLEAN_OUTPUTS = True

if CLEAN_OUTPUTS and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        print("Note: some files could not be removed (still open elsewhere); continuing anyway.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRAWN_FIGURES = {}


def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Choose the station

NRT products are open data: unlike the Level 2 notebook, no authentication is needed.

In [ ]:
stations = meta.list_stations(ATMO_STATION)
print(f"{len(stations)} ATMO stations available.")

stations_by_code = {s.id: s for s in stations}

def _country_options():
    codes = sorted({s.country_code for s in stations if s.country_code})
    return [("All countries", "ALL")] + [(c, c) for c in codes]

def _stations_for(country):
    sel = [s for s in stations if country == "ALL" or s.country_code == country]
    return sorted((f"{s.id} - {s.name} ({s.country_code})", s.id) for s in sel)

# Preserve an already-confirmed station if this cell is re-run
if "SELECTED_STATION" not in globals():
    SELECTED_STATION = None
_initial_code = SELECTED_STATION if SELECTED_STATION in stations_by_code else DEFAULT_STATION_CODE
if _initial_code not in stations_by_code:
    _initial_code = stations[0].id
_initial_country = stations_by_code[_initial_code].country_code or "ALL"

country_dd = widgets.Dropdown(options=_country_options(), value=_initial_country,
                              description="Country:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="320px"))
station_dd = widgets.Dropdown(options=_stations_for(_initial_country), value=_initial_code,
                              description="Station:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="620px"))
confirm_btn = widgets.Button(description="Confirm station", button_style="success", icon="check")
confirm_out = widgets.Output()
map_out = widgets.Output()

# Set while the dropdowns are repopulated programmatically, so the map is redrawn exactly once
# per user action instead of once per cascading update.
_updating = {"busy": False}

GEO_LAYOUT = dict(
    resolution=50, showcountries=True, countrycolor="#888",
    showland=True, landcolor="#f2efe9", showocean=True, oceancolor="#dbe9f2",
    showlakes=False, showcoastlines=True, coastlinecolor="#999",
)

map_out = widgets.Output()
map_area = map_out

def _draw_map_plotly(located, selected, current):
    others = [s for s in located if s.id != current]
    figure = go.Figure()
    if others:
        figure.add_trace(go.Scattergeo(
            lon=[s.lon for s in others], lat=[s.lat for s in others],
            text=[f"{s.id} - {s.name}" for s in others],
            hoverinfo="text", mode="markers", name="Stations",
            marker=dict(size=8, color="#3B4872", line=dict(width=1, color="white"))))
    if selected:
        figure.add_trace(go.Scattergeo(
            lon=[s.lon for s in selected], lat=[s.lat for s in selected],
            text=[f"{s.id} - {s.name}" for s in selected],
            hoverinfo="text", mode="markers", name="Selected",
            marker=dict(size=16, color="crimson", symbol="star",
                        line=dict(width=1.5, color="white"))))
    figure.update_geos(fitbounds="locations", **GEO_LAYOUT)
    figure.update_layout(height=420, margin=dict(l=0, r=0, t=36, b=0), showlegend=False,
                         title=dict(text=f"{len(located)} ATMO station(s) - selected: {current}",
                                    font=dict(size=13)))
    figure.show()

def _draw_map_static(located, selected, current):
    """Static alternative: a plain scatter, no basemap. Selected with MAP_BACKEND."""
    if True:
        others = [s for s in located if s.id != current]
        lons = [s.lon for s in located]
        lats = [s.lat for s in located]

        fig, ax = plt.subplots(figsize=(7, 4.5))
        if others:
            ax.scatter([s.lon for s in others], [s.lat for s in others],
                       s=45, c="#3B4872", edgecolors="white", zorder=3, label="Stations")
        if selected:
            ax.scatter([s.lon for s in selected], [s.lat for s in selected],
                       s=160, c="crimson", edgecolors="white", zorder=4, marker="*",
                       label="Selected")

        # A single far-away station (e.g. La Reunion for FR) would squash all the others into
        # an unreadable cluster, so the view is framed on the bulk of the stations.
        def _bulk_range(values):
            """Range of the values, discarding the extremes when there are enough points."""
            ordered = sorted(values)
            if len(ordered) < 5:
                return ordered[0], ordered[-1]
            k = max(1, len(ordered) // 10)      # number of points trimmed at each end
            return ordered[k], ordered[-(k + 1)]

        lon_lo, lon_hi = _bulk_range(lons)
        lat_lo, lat_hi = _bulk_range(lats)
        for s in selected:                      # always keep the selected station in view
            lon_lo, lon_hi = min(lon_lo, s.lon), max(lon_hi, s.lon)
            lat_lo, lat_hi = min(lat_lo, s.lat), max(lat_hi, s.lat)
        pad = max(1.5, 0.15 * max(lon_hi - lon_lo, lat_hi - lat_lo))
        ax.set_xlim(lon_lo - pad, lon_hi + pad)
        ax.set_ylim(lat_lo - pad, lat_hi + pad)

        visible = [s for s in located
                   if lon_lo - pad <= s.lon <= lon_hi + pad and lat_lo - pad <= s.lat <= lat_hi + pad]
        for s in (visible if len(visible) <= 12 else selected):
            ax.annotate(s.id, (s.lon, s.lat), textcoords="offset points", xytext=(6, 4),
                        fontsize=8, zorder=5)

        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        title = f"{len(located)} ATMO station(s) - selected: {current}"
        if len(visible) < len(located):
            title += f" ({len(located) - len(visible)} outside the view)"
        ax.set_title(title, fontsize=11)
        ax.grid(True, linestyle=":", alpha=0.6)
        mean_lat = sum(lats) / len(lats)
        ax.set_aspect(1.0 / max(0.2, math.cos(math.radians(mean_lat))), adjustable="box")
        if others and selected:
            ax.legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()

def _draw_map(*_):
    shown = [stations_by_code[c] for _, c in station_dd.options]
    located = [s for s in shown if s.lat is not None and s.lon is not None]
    current = station_dd.value
    selected = [s for s in located if s.id == current]
    with map_out:
        map_out.clear_output(wait=True)
        if not located:
            print("No coordinates available for these stations.")
            return
        if MAP_BACKEND == "plotly":
            _draw_map_plotly(located, selected, current)
        else:
            _draw_map_static(located, selected, current)

def _on_country_change(change):
    opts = _stations_for(change["new"])
    _updating["busy"] = True
    try:
        station_dd.options = opts
        if opts:
            station_dd.value = opts[0][1]
    finally:
        _updating["busy"] = False
    _draw_map()

def _on_station_change(change):
    if not _updating["busy"]:
        _draw_map()

def _on_confirm(_):
    global SELECTED_STATION
    SELECTED_STATION = station_dd.value
    s = stations_by_code[SELECTED_STATION]
    with confirm_out:
        confirm_out.clear_output()
        print(f"Confirmed station: {s.id} - {s.name} ({s.country_code})")

country_dd.observe(_on_country_change, names="value")
station_dd.observe(_on_station_change, names="value")
confirm_btn.on_click(_on_confirm)

display(widgets.VBox([country_dd, station_dd, map_area, confirm_btn, confirm_out]))
_draw_map()

with confirm_out:
    if SELECTED_STATION:
        print(f"Confirmed station: {SELECTED_STATION} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_STATION_CODE}.")

## 3. Choose the two species

The two NRT products to compare. Each is looked up as
`.../cpmeta/atc<Species>NrtDataObject`; if a station does not publish one of them, the next cell
says so rather than failing.

In [ ]:
SELECTED_STATION = globals().get("SELECTED_STATION") or DEFAULT_STATION_CODE

all_data_types = meta.list_datatypes()
nrt_types = {}
for dt in all_data_types:
    m = re.search(r"/atc(.+?)NrtDataObject$", dt.uri)
    if m and dt.has_data_access:
        nrt_types[m.group(1).upper()] = dt      # CH4, CO, CO2, N2O, MTO ...

_gas_options = sorted((f"{g} - {nrt_types[g].label}", g) for g in nrt_types)
print(f"{len(nrt_types)} NRT product(s) available: {', '.join(sorted(nrt_types))}")


def _gas_default(wanted, fallback_index):
    return wanted if wanted in nrt_types else _gas_options[fallback_index][1]


gas_a = widgets.Dropdown(options=_gas_options, value=_gas_default(DEFAULT_GASES[0], 0),
                         description="First:", style={"description_width": "initial"},
                         layout=widgets.Layout(width="520px"))
gas_b = widgets.Dropdown(options=_gas_options, value=_gas_default(DEFAULT_GASES[1], -1),
                         description="Second:", style={"description_width": "initial"},
                         layout=widgets.Layout(width="520px"))
confirm_gas_btn = widgets.Button(description="Confirm species", button_style="success",
                                 icon="check")
gas_out = widgets.Output()

if "SELECTED_GASES" not in globals():
    SELECTED_GASES = None


def _on_gas_confirm(_):
    global SELECTED_GASES
    with gas_out:
        gas_out.clear_output()
        if gas_a.value == gas_b.value:
            print("The two species are the same - pick two different ones.")
            SELECTED_GASES = None
            return
        SELECTED_GASES = [gas_a.value, gas_b.value]
        print("Confirmed:", " and ".join(SELECTED_GASES))


confirm_gas_btn.on_click(_on_gas_confirm)
display(widgets.VBox([gas_a, gas_b, confirm_gas_btn, gas_out]))

with gas_out:
    print(f"Not confirmed yet - will default to {gas_a.value} and {gas_b.value}.")

## 4. Load both and merge on time

For each species, the NRT files of the selected station are listed and read. NRT comes in two
flavours — a file per day, and a growing time series covering a period — so the same measurement can
appear in both; keeping the first occurrence of each timestamp avoids counting it twice.

Columns are renamed with the species and the measurement height, then the two are merged on time.
Only timestamps present in both survive.

In [ ]:
SELECTED_GASES = globals().get("SELECTED_GASES") or [gas_a.value, gas_b.value]
station_obj = stations_by_code[SELECTED_STATION]
STATION_URI_HTTP = station_obj.uri.replace("https://", "http://")
print(f"Station: {station_obj.id} - {station_obj.name}")


def _resolve_time_column(frame):
    for name in TIME_COLUMN_CANDIDATES:
        if name in frame.columns:
            return name
    for name in frame.columns:
        if pd.api.types.is_datetime64_any_dtype(frame[name]):
            return name
    return None


def load_gas(gas):
    """Every NRT file of this species at this station, concatenated and de-duplicated."""
    dt = nrt_types[gas]
    objs = meta.list_data_objects(datatype=dt.uri, station=STATION_URI_HTTP)
    if not objs:
        print(f"  {gas}: no NRT file at this station.")
        return None, []

    parts, columns = [], []
    # Newest first: with the cap below, a partial load is then the most recent period.
    for dobj in sorted(objs, key=lambda o: o.filename, reverse=True)[:MAX_FILES_PER_GAS]:
        try:
            d = pd.DataFrame(data.get_columns_as_arrays(meta.get_dobj_meta(dobj)))
        except Exception as exc:  # noqa: BLE001 - one unreadable file must not stop the rest
            print(f"  {gas}: could not read {dobj.filename} ({type(exc).__name__})")
            continue
        tcol = _resolve_time_column(d)
        if tcol is None:
            continue
        if DEFAULT_QUALITY_FLAG in d.columns:
            before = len(d)
            d = d[d[DEFAULT_QUALITY_FLAG].isin(ACCEPTED_FLAGS)]
            if before and d.empty:
                seen = sorted(set(pd.DataFrame(data.get_columns_as_arrays(
                    meta.get_dobj_meta(dobj)))[DEFAULT_QUALITY_FLAG]))
                print(f"  {gas}: every row of {dobj.filename} was filtered out by the quality "
                      f"flag. Values present: {seen}; accepted: {list(ACCEPTED_FLAGS)}")
        d = d.rename(columns={tcol: "time"})
        d["time"] = pd.to_datetime(d["time"], errors="coerce")

        height = f"_{dobj.sampling_height:g}m" if dobj.sampling_height else ""
        keep = {c: f"{c}{height}" for c in d.columns
                if c not in ("time", "Flag", "NbPoints", "Stdev")
                and pd.api.types.is_numeric_dtype(d[c])}
        if not keep:
            continue
        parts.append(d[["time"] + list(keep)].rename(columns=keep).dropna(subset=["time"]))
        columns += [c for c in keep.values() if c not in columns]

    if not parts:
        print(f"  {gas}: {len(objs)} file(s) found, none readable as a time series.")
        return None, []

    # Daily files and the growing series overlap, so the same timestamp can arrive twice.
    merged = (pd.concat(parts, ignore_index=True)
                .sort_values("time")
                .drop_duplicates(subset=["time"], keep="first"))
    print(f"  {gas}: {len(objs)} file(s), {len(merged)} unique timestamps, columns {columns}")
    return merged, columns


print("Loading:")
frame_a, cols_a = load_gas(SELECTED_GASES[0])
frame_b, cols_b = load_gas(SELECTED_GASES[1])

if frame_a is None or frame_b is None:
    raise RuntimeError("One of the two species has no usable NRT data at this station. "
                       "Pick another species, or another station.")

def _describe(name, frame):
    print(f"  {name}: {len(frame)} rows, {frame['time'].min()} -> {frame['time'].max()}")
    print(f"     first timestamps: {[str(t) for t in frame['time'].head(3)]}")


combined = pd.merge(frame_a, frame_b, on="time", how="inner").sort_values("time")

if combined.empty:
    # Two NRT products of the same station normally cover the same period, so an empty merge
    # is almost always a matter of timestamps that do not fall on exactly the same instant -
    # a different averaging interval, or means labelled at the middle rather than the start.
    print("\nNo exactly matching timestamps. What the two records actually contain:")
    _describe(SELECTED_GASES[0], frame_a)
    _describe(SELECTED_GASES[1], frame_b)

    overlap = (max(frame_a["time"].min(), frame_b["time"].min()),
               min(frame_a["time"].max(), frame_b["time"].max()))
    if overlap[0] > overlap[1]:
        print("\nThe two periods do not overlap at all: nothing can be compared.")
    else:
        print(f"\nThe periods do overlap ({overlap[0]} -> {overlap[1]}), so the timestamps are "
              f"simply not aligned.")
        print(f"Matching each point with the nearest one within {ALIGN_TOLERANCE} ...")
        combined = pd.merge_asof(
            frame_a.sort_values("time"), frame_b.sort_values("time"),
            on="time", tolerance=pd.Timedelta(ALIGN_TOLERANCE), direction="nearest",
        ).dropna(subset=[c for c in cols_b])
        if combined.empty:
            print("Still nothing within that tolerance. Widen ALIGN_TOLERANCE in the "
                  "Configuration cell, or compare two products with the same sampling.")
        else:
            print(f"{len(combined)} points matched. Note that the two values of a pair are close "
                  f"in time but not simultaneous.")
else:
    print(f"\n{len(combined)} timestamps present in both records.")

## 5. Plot

Two species rarely share a scale, so when both are shown each gets its own axis. A single series
uses one axis, and the correlation is printed only when the two are compared.

In [ ]:
mode_dd = widgets.Dropdown(
    options=[("Both, on separate axes", "both"),
             ("First species only", "left"),
             ("Second species only", "right")],
    value="both", description="Show:", style={"description_width": "initial"},
    layout=widgets.Layout(width="360px"))
series_a = widgets.Dropdown(options=cols_a, value=cols_a[0], description="Left axis:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="420px"))
series_b = widgets.Dropdown(options=cols_b, value=cols_b[0], description="Right axis:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="420px"))
plot_btn = widgets.Button(description="Plot", button_style="success", icon="line-chart")
plot_out = widgets.Output()


def _draw(_=None):
    with plot_out:
        plot_out.clear_output()
        if combined.empty:
            print("Nothing to plot: the two records have no timestamps in common.")
            return

        mode = mode_dd.value
        series_a.disabled = mode == "right"
        series_b.disabled = mode == "left"

        fig, ax1 = plt.subplots(figsize=(13, 5))
        ax1.set_xlabel("Time")

        if mode in ("both", "left"):
            ax1.plot(combined["time"], combined[series_a.value], linewidth=0.8, color="tab:blue")
            ax1.set_ylabel(series_a.value, color="tab:blue" if mode == "both" else "black")
            if mode == "both":
                ax1.tick_params(axis="y", labelcolor="tab:blue")

        if mode == "right":
            ax1.plot(combined["time"], combined[series_b.value], linewidth=0.8, color="tab:red")
            ax1.set_ylabel(series_b.value)
        elif mode == "both":
            ax2 = ax1.twinx()
            ax2.plot(combined["time"], combined[series_b.value], linewidth=0.8, color="tab:red")
            ax2.set_ylabel(series_b.value, color="tab:red")
            ax2.tick_params(axis="y", labelcolor="tab:red")

        shown = {"both": f"{series_a.value} and {series_b.value}",
                 "left": series_a.value, "right": series_b.value}[mode]
        ax1.set_title(f"{station_obj.name} (NRT): {shown}")
        fig.tight_layout()
        register_figure(fig, f"{SELECTED_STATION}_nrt_{shown.replace(' ', '_')}")
        plt.show()



for w in (mode_dd, series_a, series_b):
    w.observe(_draw, names="value")
plot_btn.on_click(_draw)
display(widgets.VBox([mode_dd, series_a, series_b, plot_btn, plot_out]))
_draw()

<a id="export"></a>
## Keeping the results

Run the cell below to write the figure and the merged table into the `outputs/` folder next to this
notebook. As in the other notebooks, that folder is emptied at the start of each run.

In [ ]:
written = []
for stem, fig in DRAWN_FIGURES.items():
    p = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    written.append(p.name)

csv_path = OUTPUT_DIR / f"{SELECTED_STATION}_nrt_two_species.csv"
combined.to_csv(csv_path, index=False)

print(f"Figures: {', '.join(written) if written else 'none drawn yet'}")
print(f"Table  : {csv_path} ({len(combined)} rows)")
print("Download them from the file browser on the left: right-click, then Download.")